In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/asmitjain2005/raw-chunks-data-chatbot/govt_chunks_raw.jsonl
/kaggle/input/datasets/asmitjain2005/raw-chunks-data-chatbot/who_chunks_raw.jsonl


In [3]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.5 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [4]:
import json
from sentence_transformers import SentenceTransformer

print("⏳ Loading raw data from Kaggle input directory...")

# Paths to your uploaded files
govt_input_path = "/kaggle/input/datasets/asmitjain2005/raw-chunks-data-chatbot/govt_chunks_raw.jsonl"
who_input_path = "/kaggle/input/datasets/asmitjain2005/raw-chunks-data-chatbot/who_chunks_raw.jsonl"

# Read government chunks
with open(govt_input_path, 'r', encoding='utf-8') as f:
    govt_chunks_raw = [json.loads(line) for line in f]

# Read WHO chunks
with open(who_input_path, 'r', encoding='utf-8') as f:
    who_chunks_raw = [json.loads(line) for line in f]

print(f"✅ Loaded {len(govt_chunks_raw)} Government chunks.")
print(f"✅ Loaded {len(who_chunks_raw)} WHO chunks.")

⏳ Loading raw data from Kaggle input directory...
✅ Loaded 5090 Government chunks.
✅ Loaded 2233 WHO chunks.


In [5]:
import torch

print("⏳ Loading BGE-M3 embedding model onto cloud GPU...")

# Check if GPU is active
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🤖 Using device: {device.upper()}")

# Load model and explicitly move it to the GPU
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

print("✅ BGE-M3 Model loaded cleanly on cloud accelerator!")

⏳ Loading BGE-M3 embedding model onto cloud GPU...
🤖 Using device: CUDA


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-M3 Model loaded cleanly on cloud accelerator!


In [6]:
def generate_embeddings_and_save(chunks_list, output_filename):
    print(f"\n⏳ Generating embeddings for {len(chunks_list)} chunks...")
    
    texts = [chunk["text"] for chunk in chunks_list]
    
    # The model automatically runs on the GPU device specified during initialization
    embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True)
    
    for i, chunk in enumerate(chunks_list):
        chunk["embedding"] = embeddings[i].tolist()
        
    # Save directly to Kaggle's working directory
    output_filepath = f"/kaggle/working/{output_filename}"
    with open(output_filepath, 'w', encoding='utf-8') as f:
        for chunk in chunks_list:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
            
    print(f"✅ Final embedded database payload saved to: {output_filepath}")

print("--- Starting Phase 3 & 4: High-Speed Embedding Generation ---")
generate_embeddings_and_save(govt_chunks_raw, "govt_chunks_embedded.jsonl")
generate_embeddings_and_save(who_chunks_raw, "who_chunks_embedded.jsonl")

--- Starting Phase 3 & 4: High-Speed Embedding Generation ---

⏳ Generating embeddings for 5090 chunks...


Batches:   0%|          | 0/160 [00:00<?, ?it/s]

✅ Final embedded database payload saved to: /kaggle/working/govt_chunks_embedded.jsonl

⏳ Generating embeddings for 2233 chunks...


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

✅ Final embedded database payload saved to: /kaggle/working/who_chunks_embedded.jsonl
